## Workspace setup

In [1]:
from datetime import datetime 
import uproot
import awkward as ak
import tensorflow as tf
import numpy as np
import importlib
from functools import partial
from termcolor import colored

from tensorflow.data import Dataset, TFRecordDataset
from tensorflow.data.experimental import TFRecordWriter
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Example, Features, Feature
import tensorflow_datasets as tfds

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

import io_functions as io

2025-11-27 15:29:58.218006: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Convert simulated data.

Simulated data contains Track3D objects, for generated and reconstructed tracks.
We create TFRecords for both SimEvent and RecoEvent.

In [18]:
%%time
importlib.reload(io)

#dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
#rooFfiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC_2025-10-02T09-23.root:TPCData']

fileName = 'SimEvent_Track3D_TwoProng_gun_MC_2025-11-28T09-40.root'
dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build_with_TF/resources/'
#rootFiles = [dataPath+fileName+':TPCData']

rootFiles = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC_2025-11-28T16-37_100k.root'+':TPCData',
             dataPath+'SimEvent_Track3D_TwoProng_gun_MC_100k.root'+':TPCData']

# Convert ROOT files to TF format and save to output directory
simOutputDir = 'SimEvent_Track3D_TwoProng_gun_MC'
io.convertROOT(rootFiles, simOutputDir, fields= io.simEventFields)

# uproot always takes the first branch with given name, unless 
# explicit branch is given as input path. Filtering by branches in
# iterate does not work. We have to give full path to the branch: TPCData/RecoEvent
rootFiles = [aFile+'/RecoEvent' for aFile in rootFiles]

# Convert ROOT files to TF format and save to output directory
recoOutputDir = 'RecoEvent_Track3D_TwoProng_gun_MC'
io.convertROOT(rootFiles, recoOutputDir, fields=io.recoEventFields)

CPU times: user 41min 1s, sys: 5min 7s, total: 46min 8s
Wall time: 24min 46s


### Merge SimEvent and RecoEvent data


In [19]:
# merge SimEvent and RecoEvent data

simOutputDir = 'SimEvent_Track3D_TwoProng_gun_MC'
recoOutputDir = 'RecoEvent_Track3D_TwoProng_gun_MC'

simDataset = tf.data.Dataset.load(simOutputDir, compression="GZIP")
recoDataset = tf.data.Dataset.load(recoOutputDir, compression="GZIP")

mergedDataset = tf.data.Dataset.zip((simDataset, recoDataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco
    }
)

# filter merged dataset
# calculate sim alpha length and select events with length > 30 mm
minAlphaLength = 30.0
mergedDataset = mergedDataset.filter(
    lambda x: tf.reduce_all(
        tf.greater(
            tf.sqrt(
                tf.math.reduce_sum(tf.square(x['sim'][1][:,0] - x['sim'][1][:,1]), axis=-1)
            ), minAlphaLength
        )
    )
)

# calculate sim alpha length and select events with length < 100 mm
maxAlphaLength = 10000.0
mergedDataset = mergedDataset.filter(
    lambda x: tf.reduce_all(
        tf.less(
            tf.sqrt(
                tf.math.reduce_sum(tf.square(x['sim'][1][:,0] - x['sim'][1][:,1]), axis=-1)
            ), maxAlphaLength
        )
    )
)

# save merged dataset
outputDir = 'MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm'
mergedDataset.save(outputDir, compression="GZIP")

### Create a dataset and run benchmark

In [24]:
import io_functions as io
importlib.reload(io)

import plotting_functions as plf
importlib.reload(plf)

batchSize = 32
#dataset = tf.data.Dataset.load('SimEvent_Track3D_TwoProng_gun_MC', compression="GZIP")
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm', compression="GZIP")
dataset = dataset.batch(batchSize, drop_remainder=True)
#dataset = dataset.map(lambda x,y: (tf.reshape(x, (-1,)+io.projections.shape), tf.reshape(y, (-1,9))))
dataset = dataset.take(500).cache().prefetch(tf.data.AUTOTUNE)

print(colored("Timing before caching.","blue"))
tfds.benchmark(dataset,batch_size=batchSize)
print(colored("Timing after caching.", "blue"))
tfds.benchmark(dataset,batch_size=batchSize)

Timing before caching.

************ Summary ************



  0%|          | 0/500 [00:00<?, ?it/s]

Examples/sec (First included) 286.11 ex/sec (total: 16032 ex, 56.03 sec)
Examples/sec (First only) 11.46 ex/sec (total: 32 ex, 2.79 sec)
Examples/sec (First excluded) 300.52 ex/sec (total: 16000 ex, 53.24 sec)
Timing after caching.

************ Summary ************



2025-12-01 11:43:10.154635: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


  0%|          | 0/500 [00:00<?, ?it/s]

Examples/sec (First included) 160223.14 ex/sec (total: 16032 ex, 0.10 sec)
Examples/sec (First only) 5278.78 ex/sec (total: 32 ex, 0.01 sec)
Examples/sec (First excluded) 170215.57 ex/sec (total: 16000 ex, 0.09 sec)


2025-12-01 11:43:10.256406: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,duration,num_examples,avg
first+lasts,0.100060,16032,160223.137102
first,0.006062,32,5278.783267
lasts,0.093998,16000,170215.569082


### Load and convert real data events

MC data events contain reconstructed Track3D in both trees: RecoEvent and SimEvent, while real data events have only in RecoEvent.
We load the RecoEvent and put the data twice into the dict to maintain the same structure as for simulated data.

In [ ]:
%%time
importlib.reload(io)

dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
# uproot always takes the first branch with given name, unless 
# explicit branch is given as input path. Filtering by branches in
# iterate does not work. 
rootfiles = [dataPath+'RecoEvent_TwoProng_2022-04-12T08-03-44.root:TPCData']
outputDir = 'RecoEvent_Track3D_TwoProng_2022-04-12T08-03-44'

# Convert ROOT files to TF format and save to output directory
# use the simEventFields which contain the Track3D and the event data (images)
io.convertROOT(rootfiles, outputDir, fields=io.simEventFields)

dataset = tf.data.Dataset.load(outputDir, compression="GZIP")

mergedDataset = tf.data.Dataset.zip((dataset, dataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco
    }
)

# filter merged dataset
# calculate sim alpha length and select events with length > 20 mm
minAlphaLength = 20.0
mergedDataset = mergedDataset.filter(
    lambda x: tf.reduce_all(
        tf.greater(
            tf.sqrt(
                tf.math.reduce_sum(tf.square(x['sim'][1][:,0] - x['sim'][1][:,1]), axis=-1)
            ), minAlphaLength
        )
    )
)

# calculate sim alpha length and select events with length < 100 mm
maxAlphaLength = 100.0
mergedDataset = mergedDataset.filter(
    lambda x: tf.reduce_all(
        tf.less(
            tf.sqrt(
                tf.math.reduce_sum(tf.square(x['sim'][1][:,0] - x['sim'][1][:,1]), axis=-1)
            ), maxAlphaLength
        )
    )
)

# save merged dataset
outputDir = 'MergedEvent_Track3D_TwoProng_2022-04-12T08-03-44'
mergedDataset.save(outputDir, compression="GZIP")